# Road Following — Interactive Regression (Real JetRacer)

Collect data, train, and test the ResNet-18 model directly on the **real JetRacer** with a CSI/USB camera.

| Step | Description |
|------|-------------|
| 1 | Start CSI camera |
| 2 | Configure Task & Dataset |
| 3 | Collect data (click image to label) |
| 4 | Select model architecture |
| 5 | Live prediction preview |
| 6 | Train / Evaluate |
| 7 | All-in-one integrated UI |

### 1. Camera

In [ ]:
from jetcam.csi_camera import CSICamera
# from jetcam.usb_camera import USBCamera

camera = CSICamera(width=224, height=224)
# camera = USBCamera(width=224, height=224)

camera.running = True
print(f"Camera started: {camera.width}x{camera.height}")


### 2. Task & Dataset

In [ ]:
import torchvision.transforms as transforms
from xy_dataset import XYDataset

TASK = 'road_following'

CATEGORIES = ['apex']

DATASETS = ['A', 'B']

TRANSFORMS = transforms.Compose([
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

datasets = {}
for name in DATASETS:
    datasets[name] = XYDataset(TASK + '_' + name, CATEGORIES, TRANSFORMS, random_hflip=True)

print(f"Task: {TASK}")
print(f"Categories: {CATEGORIES}")
print(f"Datasets: {list(datasets.keys())}")


### 3. Data Collection

**Click on the camera image** to save a labeled data sample with $(x, y)$ coordinates at the clicked point.
A green circle will appear on the snapshot to confirm the labeled point.

In [ ]:
import cv2
import ipywidgets
import traitlets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg
from jupyter_clickable_image_widget import ClickableImageWidget

# Initialize active dataset
dataset = datasets[DATASETS[0]]

# Unobserve all callbacks from camera in case we are running this cell for second time
camera.unobserve_all()

# Create image preview — ClickableImageWidget supports click events for labeling
camera_widget   = ClickableImageWidget(width=camera.width, height=camera.height)
snapshot_widget = ipywidgets.Image(width=camera.width, height=camera.height)
traitlets.dlink((camera, 'value'), (camera_widget, 'value'), transform=bgr8_to_jpeg)

# Dataset & category selector
dataset_widget  = ipywidgets.Dropdown(options=DATASETS, description='dataset')
category_widget = ipywidgets.Dropdown(options=CATEGORIES, description='category')

# Count display
count_widget = ipywidgets.IntText(description='count', value=len(dataset))

def update_counts(change):
    count_widget.value = dataset.get_count(category_widget.value)

def set_dataset(change):
    global dataset
    dataset = datasets[change['new']]
    count_widget.value = len(dataset)

dataset_widget.observe(set_dataset, names='value')
category_widget.observe(update_counts, names='value')

def save_snapshot(_, content, msg):
    if content['event'] == 'click':
        data = content['eventData']
        x = data['offsetX']
        y = data['offsetY']

        # Save labeled entry to disk
        dataset.save_entry(category_widget.value, camera.value, x, y)

        # Display saved snapshot with green dot confirming the label
        snapshot = camera.value.copy()
        snapshot = cv2.circle(snapshot, (x, y), 8, (0, 255, 0), 3)
        snapshot_widget.value = bgr8_to_jpeg(snapshot)
        count_widget.value = dataset.get_count(category_widget.value)

camera_widget.on_msg(save_snapshot)

data_collection_widget = ipywidgets.VBox([
    ipywidgets.HBox([camera_widget, snapshot_widget]),
    dataset_widget,
    category_widget,
    count_widget
])

display(data_collection_widget)


### 4. Model

In [ ]:
import torch
import torchvision

device = torch.device('cuda')
output_dim = 2 * len(dataset.categories)  # x, y coordinate for each category

# ALEXNET
# model = torchvision.models.alexnet(pretrained=True)
# model.classifier[-1] = torch.nn.Linear(4096, output_dim)

# SQUEEZENET
# model = torchvision.models.squeezenet1_1(pretrained=True)
# model.classifier[1] = torch.nn.Conv2d(512, output_dim, kernel_size=1)
# model.num_classes = len(dataset.categories)

# RESNET 18
model = torchvision.models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, output_dim)

# RESNET 34
# model = torchvision.models.resnet34(pretrained=True)
# model.fc = torch.nn.Linear(512, output_dim)

# DENSENET 121
# model = torchvision.models.densenet121(pretrained=True)
# model.classifier = torch.nn.Linear(model.num_features, output_dim)

model = model.to(device)

# Model save / load
model_save_button = ipywidgets.Button(description='save model', button_style='success', icon='save')
model_load_button = ipywidgets.Button(description='load model', button_style='info',    icon='upload')
model_path_widget = ipywidgets.Text(description='model path', value='road_following_model.pth')

def load_model(c):
    model.load_state_dict(torch.load(model_path_widget.value))
    print(f"Model loaded from '{model_path_widget.value}'")
model_load_button.on_click(load_model)

def save_model(c):
    torch.save(model.state_dict(), model_path_widget.value)
    print(f"Model saved to '{model_path_widget.value}'")
model_save_button.on_click(save_model)

model_widget = ipywidgets.VBox([
    model_path_widget,
    ipywidgets.HBox([model_load_button, model_save_button])
])

display(model_widget)


### 5. Live Execution

Live preview of model prediction output on the camera stream. The **red dot** is the target point the car will steer towards.

In [ ]:
import threading
import time
from utils import preprocess
import torch.nn.functional as F
import PID

state_widget      = ipywidgets.ToggleButtons(options=['stop', 'live'], description='state', value='stop')
prediction_widget = ipywidgets.Image(format='jpeg', width=camera.width, height=camera.height)

def live(state_widget, model, camera, prediction_widget):
    global dataset
    while state_widget.value == 'live':
        image = camera.value  # BGR numpy array from CSI/USB camera
        preprocessed = preprocess(image)
        
        with torch.no_grad():
            output = model(preprocessed).detach().cpu().numpy().flatten()
        
        category_index = dataset.categories.index(category_widget.value)
        x = output[2 * category_index]
        y = output[2 * category_index + 1]

        # Convert normalized [-1,1] coords to pixel coords
        x = int(camera.width  * (x / 2.0 + 0.5))
        y = int(camera.height * (y / 2.0 + 0.5))

        # Draw red prediction dot on live frame
        prediction = image.copy()
        prediction = cv2.circle(prediction, (x, y), 8, (255, 0, 0), 3)
        prediction_widget.value = bgr8_to_jpeg(prediction)

def start_live(change):
    if change['new'] == 'live':
        execute_thread = threading.Thread(target=live, args=(state_widget, model, camera, prediction_widget))
        execute_thread.start()

state_widget.observe(start_live, names='value')

live_execution_widget = ipywidgets.VBox([
    prediction_widget,
    state_widget
])

display(live_execution_widget)


### 6. Training & Evaluation

In [ ]:
BATCH_SIZE    = 8
LEARNING_RATE = 1e-3
MOMENTUM      = 0.9

optimizer = torch.optim.Adam(model.parameters())
# optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)

epochs_widget   = ipywidgets.IntText(description='epochs',   value=1)
eval_button     = ipywidgets.Button(description='evaluate',  button_style='info')
train_button    = ipywidgets.Button(description='train',     button_style='warning')
loss_widget     = ipywidgets.FloatText(description='loss')
progress_widget = ipywidgets.FloatProgress(min=0.0, max=1.0, description='progress')

def train_eval(is_training):
    global BATCH_SIZE, LEARNING_RATE, MOMENTUM, model, dataset, optimizer

    try:
        train_loader = torch.utils.data.DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        state_widget.value   = 'stop'
        train_button.disabled = True
        eval_button.disabled  = True
        time.sleep(1)

        model.train() if is_training else model.eval()

        while epochs_widget.value > 0:
            i         = 0
            sum_loss  = 0.0

            for images, category_idx, xy in iter(train_loader):
                images = images.to(device)
                xy     = xy.to(device)

                if is_training:
                    optimizer.zero_grad()

                outputs = model(images)

                # MSE loss over (x, y) for each sample's associated category
                loss = 0.0
                for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                    loss += torch.mean(
                        (outputs[batch_idx][2 * cat_idx:2 * cat_idx + 2] - xy[batch_idx]) ** 2
                    )
                loss /= len(category_idx)

                if is_training:
                    loss.backward()
                    optimizer.step()

                count = len(category_idx.flatten())
                i        += count
                sum_loss += float(loss)
                progress_widget.value = i / len(dataset)
                loss_widget.value     = sum_loss / i

            if is_training:
                epochs_widget.value -= 1
            else:
                break

    except Exception as e:
        print(f"Error during {'training' if is_training else 'evaluation'}: {e}")

    model.eval()
    train_button.disabled = False
    eval_button.disabled  = False
    state_widget.value    = 'live'

train_button.on_click(lambda c: train_eval(is_training=True))
eval_button.on_click(lambda c: train_eval(is_training=False))

train_eval_widget = ipywidgets.VBox([
    epochs_widget,
    progress_widget,
    loss_widget,
    ipywidgets.HBox([train_button, eval_button])
])

display(train_eval_widget)


### 7. All Together!

Fully integrated interface — Data collection, Live preview, Training and Model saving.

| Widget | Description |
|--------|-------------|
| dataset | Select active dataset (A / B) |
| category | Select active category to label |
| epochs | Number of training epochs |
| train | Train model on the active dataset |
| evaluate | Evaluate loss on the active dataset |
| model path | Path to the `.pth` model file |
| load model | Load weights from file |
| save model | Save weights to file |
| stop | Disable live preview |
| live | Enable live model output preview |

In [ ]:
all_widget = ipywidgets.VBox([
    ipywidgets.HBox([data_collection_widget, live_execution_widget]),
    train_eval_widget,
    model_widget
])

display(all_widget)
